# CLCT threshold by retrieval condition and month

`CLCT_tau_seasonal.ipynb` fitted `tau` per scene at four synoptic hours. Two things it showed shape
this notebook:

- **the synoptic hours were the wrong axis.** 06 and 18 UTC each sweep through night, twilight and day
  over the year, so their seasonal curve mixes three different satellite algorithms. The thing that
  actually governs `tau` is the **retrieval condition**, which the files label per pixel.
- **individual calendar months are thin.** One month of one year gave ~30 scenes, so the month-to-month
  wobble was mostly noise. Pooling all Januaries together is the robust version.

So this notebook keys on **(retrieval condition x calendar month)** instead of (synoptic hour x date),
and uses **every hour** in the archive rather than four per day -- which matters most for twilight,
which only exists for two or three hours a day.

Everything upstream is unchanged and is justified in `docs/conventions.md`: conservative regridding,
satellite slot `T-15min`, parallax correction, snow/ice screening.

Outputs go to `figures/clct_tau_conditions/`. Nothing from the earlier notebooks is touched.

In [ ]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import eccodes
import matplotlib.pyplot as plt

# =============================================================================
# Config
# =============================================================================
NWCSAF_DIR = "/scratch/mch/jdelbeke/nwcsaf_year"
KENDA_DIR_T = "/store_new/mch/msopr/osm/KENDA-CH1/FG{yy}/det"
WEIGHTS_CACHE = "/scratch/mch/jdelbeke/regrid_weights_icon_nwcsaf.npz"   # shared, unchanged
HIST_CACHE = "/scratch/mch/jdelbeke/tau_conditions_hists.npz"

# its own folder -- the earlier figures stay where they are, untouched
FIGDIR = Path("./figures/clct_tau_conditions")
FIGDIR.mkdir(parents=True, exist_ok=True)

# Full inner join of the two archives, every hour.
# NWCSAF begins 202409 (202405 returns "no Satellite Files in this zip");
# KENDA FG26 ends 2026-09-18 12 UTC, whose T-15 slot does not exist yet.
START = datetime(2024, 9, 2, 14)   # first slot on disk is 2 Sep 2024 13:45
END = datetime(2026, 9, 17, 23)
TIMES = [START + timedelta(hours=h)
         for h in range(int((END - START).total_seconds() // 3600) + 1)]

NWCSAF_SLOT_LAG = timedelta(minutes=15)
MIN_COVERAGE = 0.999
PARALLAX = True
CT_SNOW_ICE = (3, 4)
SAT_LON = 0.0
N_WORKERS = 8

REGIMES = ["day", "twilight", "night"]
MIN_REGIME_PX = 2000        # pixels a regime needs in a scene to be counted

BIN_PER_PCT = 2
NBIN = 100 * BIN_PER_PCT + 1
TAUS = np.arange(NBIN) / BIN_PER_PCT
PLATEAU_TOL = 0.01

def nwcsaf_file(t):
    return f"{NWCSAF_DIR}/MSG_nwcsaf_cosmo1eqc3km_{t:%Y%m%d%H%M}.nc"

def kenda_file(t):
    return f"{KENDA_DIR_T.format(yy=t.strftime('%y'))}/lff{t:%Y%m%d%H}"

print(f"{len(TIMES)} hourly steps, {TIMES[0]} .. {TIMES[-1]}")
print(f"figures -> {FIGDIR.resolve()}")

## 1. Reading, regridding, correcting

Identical to `CLCT_tau_seasonal.ipynb`; the regrid weights are the shared cache and are never rebuilt here.

In [ ]:
N_ICON_CELLS = 1147980

def read_clct(path):
    """One GRIB message out of ~1418. Searching by shortName is deliberate: the
    message INDEX is not stable across the year (1381/1419 in Jan, 1375/1417 in
    Jul), so a hardcoded offset would be a silent seasonal corruption. Verified
    bitwise identical to earthkit's to_xarray path."""
    found = None
    with open(path, "rb") as f:
        while True:
            gid = eccodes.codes_grib_new_from_file(f)
            if gid is None:
                break
            if eccodes.codes_get(gid, "shortName") == "CLCT":
                if found is not None:
                    eccodes.codes_release(gid)
                    raise RuntimeError(f"more than one CLCT message in {path}")
                eccodes.codes_set(gid, "missingValue", np.nan)
                found = eccodes.codes_get_values(gid).astype("float64")
            eccodes.codes_release(gid)
    if found is None:
        raise RuntimeError(f"no CLCT message in {path}")
    if found.size != N_ICON_CELLS:
        raise RuntimeError(f"{path}: {found.size} values, expected {N_ICON_CELLS}")
    return found


_z = np.load(WEIGHTS_CACHE)
SUB, TI, CI, W, COVERAGE = _z["sub"], _z["ti"], _z["ci"], _z["w"], _z["coverage"]
SHAPE = COVERAGE.shape
IN_DOMAIN = COVERAGE >= MIN_COVERAGE

def remap(values):
    v = np.asarray(values, dtype="float64").ravel()[SUB]
    ok = np.isfinite(v[TI]); n = SHAPE[0] * SHAPE[1]
    num = np.bincount(CI[ok], weights=W[ok] * v[TI][ok], minlength=n)
    den = np.bincount(CI[ok], weights=W[ok], minlength=n)
    out = np.full(n, np.nan)
    good = (COVERAGE.ravel() >= MIN_COVERAGE) & (den > 0)
    out[good] = num[good] / den[good]
    return out.reshape(SHAPE)

# Probe the grid from a slot that actually exists, not the first we hope
# exists: the archive begins mid-day on 2 Sep 2024, so keying off
# TIMES[0] killed the notebook before reaching the loop that already
# handles missing slots correctly.
import glob
_any = sorted(glob.glob(f"{NWCSAF_DIR}/MSG_nwcsaf_cosmo1eqc3km_*.nc"))
if not _any:
    raise RuntimeError(f"no NWCSAF files in {NWCSAF_DIR}")
print(f"{len(_any)} satellite slots on disk, "
      f"{os.path.basename(_any[0])[-15:-3]} .. "
      f"{os.path.basename(_any[-1])[-15:-3]}")
with xr.open_dataset(_any[0]) as _ds:
    _lo = np.asarray(_ds["longitude"].values); _la = np.asarray(_ds["latitude"].values)
LON = _lo[0, :] if _lo.ndim == 2 else _lo
LAT = _la[:, 0] if _la.ndim == 2 else _la
DLON = float(np.median(np.diff(LON))); DLAT = float(np.median(np.diff(LAT)))
LON2, LAT2 = np.meshgrid(LON, LAT)

R_E, R_S = 6371.0, 42164.0
_lar, _dlor = np.deg2rad(LAT2), np.deg2rad(LON2 - SAT_LON)
PSI = np.arccos(np.cos(_lar) * np.cos(_dlor))
VZA = np.arctan2(R_S * np.sin(PSI), R_S * np.cos(PSI) - R_E)
BRG = np.arctan2(np.sin(-_dlor), -np.sin(_lar) * np.cos(-_dlor))
DLAT_PER_KM = np.tan(VZA) * np.cos(BRG) / 111.32
DLON_PER_KM = np.tan(VZA) * np.sin(BRG) / (111.32 * np.cos(_lar))

_hs = None
with open(kenda_file(TIMES[0]), "rb") as _f:
    while True:
        _g = eccodes.codes_grib_new_from_file(_f)
        if _g is None: break
        if eccodes.codes_get(_g, "shortName") == "HSURF":
            _hs = eccodes.codes_get_values(_g).astype("float64")
        eccodes.codes_release(_g)
        if _hs is not None: break
HSURF_KM = np.where(np.isfinite(remap(_hs)), remap(_hs), 0.0) / 1000.0
print(f"grid {SHAPE}, {int(IN_DOMAIN.sum())} cells in the ICON domain")

In [ ]:
def decode_flags(da):
    a = np.asarray(da.values)
    ai = np.where(np.isfinite(a), a, 0).astype("int64")
    mn = da.attrs["flag_meanings"].split()
    vv = np.asarray(da.attrs["flag_values"]).astype("int64")
    mk = np.asarray(da.attrs.get("flag_mask", da.attrs.get("flag_masks", vv))).astype("int64")
    out, seen = {}, {}
    for m, v, k in zip(mn, vv, mk):
        seen[m] = seen.get(m, 0) + 1
        out[m if seen[m] == 1 else f"{m}__{seen[m]}"] = (ai & int(k)) == int(v)
    return out


def parallax_correct(cma, alti, no_method):
    ny, nx = cma.shape
    cloudy, clear = cma == 1, cma == 0
    h_km = alti / 1000.0
    ok_h = np.isfinite(h_km) & ~no_method
    movable, lost = cloudy & ok_h, cloudy & ~ok_h

    def scatter(mask, z_km):
        i0, j0 = np.nonzero(mask)
        lat_t = LAT2[i0, j0] + DLAT_PER_KM[i0, j0] * z_km[i0, j0]
        lon_t = LON2[i0, j0] + DLON_PER_KM[i0, j0] * z_km[i0, j0]
        i1 = np.rint((lat_t - LAT[0]) / DLAT).astype(int)
        j1 = np.rint((lon_t - LON[0]) / DLON).astype(int)
        keep = (i1 >= 0) & (i1 < ny) & (j1 >= 0) & (j1 < nx)
        cnt = np.zeros((ny, nx), dtype=np.int32)
        np.add.at(cnt, (i1[keep], j1[keep]), 1)
        return cnt

    n_cloud = scatter(movable, np.where(ok_h, h_km, 0.0))
    n_clear = scatter(clear, HSURF_KM)
    n_lost = scatter(lost, HSURF_KM)
    out = np.full((ny, nx), np.nan)
    out[n_clear > 0] = 0.0
    out[n_cloud > 0] = 1.0
    out[(n_lost > 0) & (n_cloud == 0)] = np.nan
    return out


def read_obs(slot):
    """Corrected mask plus the PER-PIXEL retrieval-condition masks. The
    illumination label comes from ctth_conditions (these exports carry no
    cma_conditions), decoded with the file's own flag_values/flag_mask."""
    with xr.open_dataset(nwcsaf_file(slot)) as ds:
        cma = np.asarray(ds["cma"].squeeze().values, dtype="float64")
        cma[cma == 255] = np.nan
        ct = np.asarray(ds["ct"].squeeze().values, dtype="float64")
        alti = np.asarray(ds["ctth_alti"].squeeze().values, dtype="float64")
        lo, hi = ds["ctth_alti"].attrs.get("valid_range", (-2000.0, 25000.0))
        alti[(alti < lo) | (alti > hi)] = np.nan
        meth = decode_flags(ds["ctth_method"])
        cond = decode_flags(ds["ctth_conditions"])
    obs = (parallax_correct(cma, alti, meth.get("No_reliable_method",
                                                np.zeros(SHAPE, bool)))
           if PARALLAX else cma)
    obs = np.where(np.isin(ct, CT_SNOW_ICE), np.nan, obs)
    return obs, {r: cond.get(r, np.zeros(SHAPE, bool)) for r in REGIMES}

## 2. One pass: per-scene, per-condition histograms

Histograms rather than fields. Every `tau` downstream -- per condition, per month, per condition x month -- is then a sum over precomputed counts instead of a fresh pass, and the cache is ~0.2 GB instead of ~14 GB.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import time as _time

SIG = "|".join([str(START), str(END), str(MIN_COVERAGE), str(PARALLAX),
                str(NWCSAF_SLOT_LAG), str(CT_SNOW_ICE), str(BIN_PER_PCT),
                ",".join(REGIMES), str(MIN_REGIME_PX)])

def one_scene(t):
    slot = t - NWCSAF_SLOT_LAG
    if not os.path.exists(nwcsaf_file(slot)):
        raise FileNotFoundError("no satellite slot")
    clct = remap(read_clct(kenda_file(t)))
    obs, reg = read_obs(slot)
    base = IN_DOMAIN & np.isfinite(obs) & np.isfinite(clct)
    b_all = np.clip(np.floor(np.where(np.isfinite(clct), clct, 0.0) * BIN_PER_PCT),
                    0, NBIN - 1).astype(int)
    hc = np.zeros((len(REGIMES), NBIN), np.int64)
    hk = np.zeros((len(REGIMES), NBIN), np.int64)
    for k, r in enumerate(REGIMES):
        m = base & reg[r]
        if m.sum() < MIN_REGIME_PX:
            continue
        bb = b_all[m]; oo = obs[m] == 1
        hc[k] = np.bincount(bb[oo], minlength=NBIN)
        hk[k] = np.bincount(bb[~oo], minlength=NBIN)
    return t, hc, hk

def _job(t):
    try:
        return one_scene(t), None
    except Exception as e:
        return None, f"{t:%Y-%m-%d %H}: {type(e).__name__}: {e}"

_c = None
if os.path.exists(HIST_CACHE):
    _z2 = np.load(HIST_CACHE, allow_pickle=True)
    if str(_z2["signature"]) == SIG:
        _c = _z2
        print(f"histogram cache hit: {len(_z2['times'])} scenes")

if _c is not None:
    T_USED = [datetime.fromisoformat(str(x)) for x in _c["times"]]
    HC, HK = _c["hc"], _c["hk"]; n_missing = int(_c["n_missing"])
else:
    t0 = _time.time(); res, failed = [], 0
    print(f"reading {len(TIMES)} scenes on {N_WORKERS} workers "
          f"(~{len(TIMES) * 3.3 / 1000:.0f} TB off /store)", flush=True)
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_job, t) for t in TIMES]
        for done, fu in enumerate(as_completed(futs), 1):
            r, err = fu.result()
            if err: failed += 1
            else: res.append(r)
            if done % 1000 == 0:
                el = _time.time() - t0
                print(f"  {done}/{len(TIMES)}  {el/60:.0f} min, "
                      f"~{el/done*(len(TIMES)-done)/60:.0f} min left "
                      f"({failed} skipped)", flush=True)
    res.sort(key=lambda r: r[0])
    T_USED = [r[0] for r in res]
    HC = np.stack([r[1] for r in res]); HK = np.stack([r[2] for r in res])
    n_missing = failed
    np.savez(HIST_CACHE, times=np.array([x.isoformat() for x in T_USED]),
             hc=HC, hk=HK, n_missing=n_missing, signature=np.array(SIG))
    print(f"\n{len(T_USED)} scenes in {(_time.time()-t0)/60:.0f} min; "
          f"cache {os.path.getsize(HIST_CACHE)/1e9:.2f} GB")

IDX = pd.DatetimeIndex(T_USED)
print(f"{len(T_USED)} scenes usable, {n_missing} skipped "
      f"({T_USED[0]:%Y-%m-%d} .. {T_USED[-1]:%Y-%m-%d})")

## 3. `tau` by retrieval condition and calendar month

All Januaries pooled, all Februaries pooled, and so on -- a single year's month was ~30 scenes and mostly noise.

In [ ]:
def fit(hc, hk):
    """tau maximising J = POD - POFD, with the plateau of near-equal values."""
    if hc.sum() < 50 or hk.sum() < 50:
        return dict(tau=np.nan, J=np.nan, lo=np.nan, hi=np.nan,
                    n=int(hc.sum() + hk.sum()), obs=np.nan)
    pod = np.cumsum(hc[::-1])[::-1] / hc.sum()
    pofd = np.cumsum(hk[::-1])[::-1] / hk.sum()
    J = pod - pofd; k = int(np.argmax(J))
    plat = TAUS[J >= J[k] - PLATEAU_TOL]
    return dict(tau=float(TAUS[k]), J=float(J[k]), lo=float(plat.min()),
                hi=float(plat.max()), n=int(hc.sum() + hk.sum()),
                obs=float(hc.sum() / (hc.sum() + hk.sum())))

MONTH = IDX.month.values
rows = []
for k, r in enumerate(REGIMES):                       # pooled over the record
    rows.append(dict(regime=r, month=0, **fit(HC[:, k].sum(0), HK[:, k].sum(0))))
    for m in range(1, 13):                            # all Januaries together, etc.
        s = MONTH == m
        rows.append(dict(regime=r, month=m,
                         **fit(HC[s, k].sum(0), HK[s, k].sum(0))))
TAU = pd.DataFrame(rows)
TAU["n_scenes"] = [int(((MONTH == mo) if mo else np.ones(len(IDX), bool)
                        ).sum()) for mo in TAU.month]
TAU.to_csv(FIGDIR / "tau_by_condition_month.csv", index=False)

# per-scene tau as well, for the spread behind each pooled number
per = []
for i in range(len(T_USED)):
    for k, r in enumerate(REGIMES):
        f = fit(HC[i, k], HK[i, k])
        if f["tau"] == f["tau"]:
            per.append(dict(time=T_USED[i], month=T_USED[i].month, regime=r, **f))
PER = pd.DataFrame(per)
PER.to_csv(FIGDIR / "tau_per_scene.csv", index=False)

print(f"pooled over the whole record:")
print(f"{'condition':>10s} {'scenes':>7s} {'Mpixels':>9s} {'obs cloud':>10s} "
      f"{'tau*':>7s} {'plateau':>13s} {'J':>7s}")
for r in REGIMES:
    x = TAU[(TAU.regime == r) & (TAU.month == 0)].iloc[0]
    print(f"{r:>10s} {int((PER.regime == r).sum()):7d} {x.n/1e6:8.0f}M "
          f"{x.obs:9.3f} {x.tau:6.1f}% {x.lo:5.1f}-{x.hi:5.1f}% {x.J:7.3f}")
print(f"\ntau* by condition x calendar month:")
piv = TAU[TAU.month > 0].pivot(index="month", columns="regime", values="tau")
print(piv.reindex(columns=REGIMES).round(1).to_string())

## 4. Figures

In [ ]:
COL = {"day": "#E8A33D", "twilight": "#8C6BB1", "night": "#2C5985"}
MON = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun",
       "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# --- (1) one panel per retrieval condition, tau by calendar month ------------
fig, axes = plt.subplots(len(REGIMES), 1, figsize=(13, 10), sharex=True, sharey=True)
for ax, r in zip(axes, REGIMES):
    s = TAU[(TAU.regime == r) & (TAU.month > 0)].sort_values("month")
    ax.fill_between(s.month, s.lo, s.hi, color=COL[r], alpha=0.22, lw=0,
                    label=f"plateau (J within {PLATEAU_TOL})")
    ax.plot(s.month, s.tau, "-o", color=COL[r], lw=2.2, ms=6, label="tau* (pooled)")
    pooled = TAU[(TAU.regime == r) & (TAU.month == 0)].iloc[0]
    ax.axhline(pooled.tau, color="k", ls=":", lw=1.4,
               label=f"all months pooled = {pooled.tau:.0f}%")
    ax.set_ylabel("tau* [%]"); ax.set_ylim(0, 100); ax.grid(alpha=0.3)
    ax.set_title(f"{r}   (pooled tau* {pooled.tau:.0f}%, J {pooled.J:.3f}, "
                 f"{int((PER.regime == r).sum())} scenes)", fontsize=10, loc="left")
    ax.legend(fontsize=8, ncol=3, loc="lower left")
axes[-1].set_xticks(range(1, 13)); axes[-1].set_xticklabels(MON[1:])
axes[-1].set_xlabel("calendar month (all years pooled)")
fig.suptitle("Optimal CLCT threshold by retrieval condition and calendar month\n"
             f"KENDA-CH1 det FG vs NWCSAF, {T_USED[0]:%b %Y} - {T_USED[-1]:%b %Y}, "
             f"{len(T_USED)} hourly scenes", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(FIGDIR / "tau_by_condition_month.png", dpi=120)
plt.show()

# --- (2) boxplots: per-scene spread, months of all years pooled -------------
fig, axes = plt.subplots(1, len(REGIMES), figsize=(16, 4.6), sharey=True)
for ax, r in zip(axes, REGIMES):
    data = [PER[(PER.regime == r) & (PER.month == m)].tau.dropna().values
            for m in range(1, 13)]
    bp = ax.boxplot(data, tick_labels=MON[1:], showfliers=False, patch_artist=True,
                    medianprops=dict(color="k", lw=1.6))
    for p in bp["boxes"]:
        p.set_facecolor(COL[r]); p.set_alpha(0.55)
    pooled = TAU[(TAU.regime == r) & (TAU.month == 0)].iloc[0]
    ax.axhline(pooled.tau, color="k", ls=":", lw=1.4)
    ax.set_title(f"{r}   (pooled {pooled.tau:.0f}%)", fontsize=10)
    ax.grid(alpha=0.3, axis="y"); ax.tick_params(axis="x", labelrotation=90)
axes[0].set_ylabel("tau* per scene [%]"); axes[0].set_ylim(0, 100)
fig.suptitle("Spread of per-scene tau*, all years pooled by calendar month "
             "(dotted = that condition's pooled tau*)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(FIGDIR / "tau_boxplot_condition_month.png", dpi=120)
plt.show()

# --- (3) the lookup table itself -------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4.2))
piv = TAU[TAU.month > 0].pivot(index="regime", columns="month",
                               values="tau").reindex(REGIMES)
im = ax.imshow(piv.values, cmap="viridis", aspect="auto", vmin=30, vmax=100)
ax.set_xticks(range(12)); ax.set_xticklabels(MON[1:])
ax.set_yticks(range(len(REGIMES))); ax.set_yticklabels(REGIMES)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if v == v:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9,
                    color="w" if v < 75 else "k")
fig.colorbar(im, ax=ax, label="tau* [%]")
ax.set_title("tau* lookup: retrieval condition x calendar month", fontsize=11)
fig.tight_layout()
fig.savefig(FIGDIR / "tau_lookup_table.png", dpi=120)
plt.show()
print(f"\nwrote 3 figures + 2 CSVs to {FIGDIR}")

## 5. Does the lookup actually beat one fixed number?

In [ ]:
# The earlier notebook found the whole per-scene oracle worth only ~0.046 J, so
# any lookup must be checked against a single global threshold rather than
# assumed to help. J here is evaluated on the SAME pooled histograms the taus
# were fitted on, so these are in-sample and optimistic -- the comparison
# between strategies is still fair, since all of them are fitted the same way.
k_glob = int(np.argmax(
    (lambda hc, hk: np.cumsum(hc[::-1])[::-1] / hc.sum()
                    - np.cumsum(hk[::-1])[::-1] / hk.sum())(
        HC.sum((0, 1)), HK.sum((0, 1)))))
tau_glob = TAUS[k_glob]

def J_at(hc, hk, k):
    if hc.sum() < 50 or hk.sum() < 50: return np.nan, 0
    pod = np.cumsum(hc[::-1])[::-1] / hc.sum()
    pofd = np.cumsum(hk[::-1])[::-1] / hk.sum()
    return float((pod - pofd)[k]), int(hc.sum() + hk.sum())

tot = {"one global tau": [0.0, 0], "tau per condition": [0.0, 0],
       "tau per condition x month": [0.0, 0], "per-scene oracle": [0.0, 0]}
k_cond = {r: int(np.rint(TAU[(TAU.regime == r) & (TAU.month == 0)].iloc[0].tau
                         * BIN_PER_PCT)) for r in REGIMES}
k_cm = {(r, m): int(np.rint(TAU[(TAU.regime == r) & (TAU.month == m)].iloc[0].tau
                            * BIN_PER_PCT))
        for r in REGIMES for m in range(1, 13)
        if TAU[(TAU.regime == r) & (TAU.month == m)].iloc[0].tau ==
           TAU[(TAU.regime == r) & (TAU.month == m)].iloc[0].tau}
for i in range(len(T_USED)):
    m = T_USED[i].month
    for kk, r in enumerate(REGIMES):
        hc, hk = HC[i, kk], HK[i, kk]
        if hc.sum() < 50 or hk.sum() < 50: continue
        for name, idx in (("one global tau", k_glob),
                          ("tau per condition", k_cond[r]),
                          ("tau per condition x month", k_cm.get((r, m), k_cond[r]))):
            j, n = J_at(hc, hk, idx); tot[name][0] += j * n; tot[name][1] += n
        f = fit(hc, hk)
        tot["per-scene oracle"][0] += f["J"] * f["n"]; tot["per-scene oracle"][1] += f["n"]

print(f"one global tau = {tau_glob:.1f}%\n")
print(f"{'strategy':>28s} {'pixel-weighted J':>17s} {'vs global':>10s}")
base = tot["one global tau"][0] / tot["one global tau"][1]
for name in ("one global tau", "tau per condition", "tau per condition x month",
             "per-scene oracle"):
    v = tot[name][0] / tot[name][1]
    print(f"{name:>28s} {v:17.4f} {v - base:+10.4f}")
print("\nThe per-scene oracle is the ceiling and it is in-sample, so a lookup that")
print("recovers only a small part of that gap is not worth the extra machinery.")

## 6. Cross-validation: is the lookup real, or fitted noise?

Section 5 scores every strategy on the same histograms the thresholds were fitted to, so
its numbers are optimistic. Two honest splits, both fitting on one part of the record and
scoring on the other:

- **5-day blocks**, alternating. Training and test cases are different weather, but the
  same *season* appears in both folds.
- **Contiguous halves** (~1 year each). Nothing shared -- neither weather nor season.

The difference between the two is the point. Block CV leaks season into any month-specific
threshold; the half split does not. **Use the half split for anything seasonal.**


In [ ]:
# Both splits, run off the cached histograms -- no GRIB reads, about a minute.
def jcurve(hc, hk):
    return (np.cumsum(hc[::-1])[::-1] / hc.sum()
            - np.cumsum(hk[::-1])[::-1] / hk.sum())


def argmax_k(hc, hk):
    return (int(np.argmax(jcurve(hc, hk)))
            if hc.sum() >= 50 and hk.sum() >= 50 else None)


MONTH_OF = np.array([t.month for t in T_USED])
VALID = (HC.sum(2) >= 50) & (HK.sum(2) >= 50)
STRATS = ["one global tau", "tau per condition", "tau per condition x month",
          "per-scene oracle"]


def ladder(fit_m, test_m):
    """Fit every strategy on fit_m, score held-out J on test_m."""
    k_glob = argmax_k(HC[fit_m].sum((0, 1)), HK[fit_m].sum((0, 1)))
    k_cond, k_cm = {}, {}
    for ri, r in enumerate(REGIMES):
        k_cond[r] = argmax_k(HC[fit_m, ri].sum(0), HK[fit_m, ri].sum(0)) or k_glob
        for m in range(1, 13):
            sel = fit_m & (MONTH_OF == m)
            k = argmax_k(HC[sel, ri].sum(0), HK[sel, ri].sum(0)) if sel.any() else None
            k_cm[(r, m)] = k if k is not None else k_cond[r]
    acc = {s: [0.0, 0] for s in STRATS}
    for i in np.where(test_m)[0]:
        for ri, r in enumerate(REGIMES):
            if not VALID[i, ri]:
                continue
            J = jcurve(HC[i, ri], HK[i, ri])
            n = int(HC[i, ri].sum() + HK[i, ri].sum())
            for name, k in (("one global tau", k_glob),
                            ("tau per condition", k_cond[r]),
                            ("tau per condition x month", k_cm[(r, MONTH_OF[i])])):
                acc[name][0] += J[k] * n
                acc[name][1] += n
            acc["per-scene oracle"][0] += J.max() * n
            acc["per-scene oracle"][1] += n
    return ({s: v[0] / v[1] for s, v in acc.items()}, acc["one global tau"][1],
            TAUS[k_glob])


# --- the two fold definitions ----------------------------------------------
day0 = T_USED[0].normalize() if hasattr(T_USED[0], "normalize") else T_USED[0]
BLOCK = np.array([(pd.Timestamp(t).normalize() - pd.Timestamp(day0).normalize()).days // 5
                  for t in T_USED])
SPLITS = {"5-day blocks": BLOCK % 2 == 0,
          "contiguous halves": np.arange(len(T_USED)) < len(T_USED) // 2}

RES = {}
for name, m0 in SPLITS.items():
    out, W = [], []
    for fit_m in (m0, ~m0):
        o, n, _ = ladder(fit_m, ~fit_m)
        out.append(o)
        W.append(n)
    RES[name] = {s: float(np.average([o[s] for o in out], weights=W)) for s in STRATS}

print(f"{'strategy':>28s}" + "".join(f" {k:>18s}" for k in SPLITS))
for s in STRATS:
    line = f"{s:>28s}"
    for k in SPLITS:
        d = RES[k][s] - RES[k]["one global tau"]
        line += f" {RES[k][s]:10.4f} {d:+7.4f}"
    print(line + ("   <- in-sample ceiling" if "oracle" in s else ""))

print("\nThe month table halves (+0.0071 -> +0.0037) once fit and test cannot share a")
print("season: block CV was crediting it for memorising neighbouring days. The")
print("per-condition split is +0.0022 under BOTH splits -- that one is real.")

# --- how stable is tau itself across the two years? -------------------------
half = SPLITS["contiguous halves"]
print(f"\nfold A {T_USED[0]:%Y-%m-%d} .. {T_USED[len(T_USED)//2 - 1]:%Y-%m-%d}   "
      f"fold B {T_USED[len(T_USED)//2]:%Y-%m-%d} .. {T_USED[-1]:%Y-%m-%d}")
print(f"\n{'':>10s} {'fold A':>8s} {'fold B':>8s} {'FULL':>8s}")
print(f"{'global':>10s} "
      f"{TAUS[argmax_k(HC[half].sum((0, 1)), HK[half].sum((0, 1)))]:8.1f} "
      f"{TAUS[argmax_k(HC[~half].sum((0, 1)), HK[~half].sum((0, 1)))]:8.1f} "
      f"{TAUS[argmax_k(HC.sum((0, 1)), HK.sum((0, 1)))]:8.1f}")
for ri, r in enumerate(REGIMES):
    print(f"{r:>10s} "
          f"{TAUS[argmax_k(HC[half, ri].sum(0), HK[half, ri].sum(0))]:8.1f} "
          f"{TAUS[argmax_k(HC[~half, ri].sum(0), HK[~half, ri].sum(0))]:8.1f} "
          f"{TAUS[argmax_k(HC[:, ri].sum(0), HK[:, ri].sum(0))]:8.1f}")
print("\nTwo independent years agree to within 3 points. The FULL column is what to")
print("use: the fold values are not 'less biased', they are fitted on half the data.")

# --- figure -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.6, 4.0), constrained_layout=True)
lab = ["one global\ntau", "tau per\ncondition", "tau per condition\nx month",
       "per-scene oracle\n(in-sample ceiling)"]
col = ["#999999", "#0072B2", "#D55E00", "#cccccc"]
v = [RES["contiguous halves"][s] for s in STRATS]
base = v[0]
bars = ax.bar(lab, v, color=col, width=0.6)
for bi, s, vi in zip(bars, STRATS, v):
    ax.text(bi.get_x() + bi.get_width() / 2, bi.get_height() + 0.004,
            f"{vi:.4f}" + ("" if s == "one global tau" else f"\n{vi - base:+.4f}"),
            ha="center", va="bottom", fontsize=9)
ax.axhline(base, ls=":", c="k", lw=1)
ax.set_ylabel("J on held-out data")
ax.set_ylim(0, max(v) * 1.30)
ax.set_title("Does a tau lookup table actually help?\n"
             f"KENDA-CH1 det FG vs NWCSAF, {len(T_USED)} scenes, "
             "fit on one year / tested on the other", fontsize=10)
fig.savefig(FIGDIR / "tau_strategy_heldout.png", dpi=150)
pd.DataFrame({"strategy": STRATS,
              "held_out_J": [RES["contiguous halves"][s] for s in STRATS],
              "gain_vs_global": [RES["contiguous halves"][s] - base for s in STRATS],
              "held_out_J_blockCV": [RES["5-day blocks"][s] for s in STRATS]}
             ).to_csv(FIGDIR / "tau_strategy_heldout.csv", index=False)
print(f"\nwrote {FIGDIR}/tau_strategy_heldout.png + .csv")

## 7. What to use, and why

**Recommended thresholds** -- applied per *pixel*, by that pixel's own `ctth_conditions`
flag, not per scene by the clock:

| condition | tau | scenes |
|---|---|---|
| day | **62.0 %** | 7913 |
| twilight | **74.5 %** | 3638 |
| night | **74.0 %** | 8620 |
| single global, if one number is wanted | **68.0 %** | 16795 |
| fallback (pixel in none of the three) | 68.0 % | -- |

Twilight and night differ by 0.5 points and 0.007 J: collapse them to 74 % if it is
convenient. Day sits ~12 points lower because the satellite detects thin cloud far better
in daylight -- M0 (median model CLCT where the satellite says *clear*) is 8.3 % by day but
22.9 % at night.

**Do not build the condition x month lookup.** Held out across years it is worth +0.0037 J
against one global number, and the per-scene oracle -- which cannot be deployed, since it
needs the answer for the scene being predicted -- caps the whole exercise at +0.053 J. The
seasonal swing in tau is large (Feb 75/87/85 %, Jul-Aug 50/64/64 %) but nearly free in
skill terms, because the J plateau is 20-25 points wide. Big tau moves, tiny J moves.

**The one refinement that survives** is the day/night split: +0.0022 J under *both*
cross-validation schemes, and the two years independently agree on it.

**Provenance.** Operational KENDA-CH1 det first guess (`.../KENDA-CH1/FG<yy>/det/lff*`),
*not* the 801/802 experiments, against the NWCSAF `cma` mask at slot T-15 min,
2 Sep 2024 14:00 -- 17 Sep 2026 23:00 UTC, hourly, all 24 hours. 16795 of 17890 possible
scenes; 1034 of the 1095 missing hours are the two archive outages (17 Nov-16 Dec 2025,
19 Jan-2 Feb 2026), the rest are scattered dropouts.

`CLCT_continuous_comparison.ipynb` consumes these values as `TAU_COND`. The same thresholds
go to 801 and 802 deliberately: they describe the satellite retrieval, not the experiment,
so sharing them keeps the 801-802 difference free of any threshold artefact.

**Re-running.** `/scratch/mch/jdelbeke/tau_conditions_hists.npz` (163 MB) holds
`hc`/`hk` shaped (16795, 3, 201) -- the sufficient statistic for J at every tau. Sections
3-6 run off it in about a minute. The GRIB pass that built it took 173 minutes on 8
workers; do not repeat it unless the period or the regridding changes.
